# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nihaaarika/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** The research paper claims high accuracy in predicting which pages need content refreshes based on historical performance metrics.
**My Methodology Question:** Where does the label come from? I would want to know if the "needs refresh" label was determined using future data (after the refresh happened) or if it was a human-annotated label from the time of prediction. If the label comes from a future window, the model is learning from data it wouldn't have in a real deployment, which is a form of target leakage.

**Finding 2:** The paper uses a random 80/20 train-test split to validate its model.
**My Methodology Question:** Does the validation design support the claim? Since this is SEO and content performance data, it is time-dependent. A random split allows future data to leak into the training set. A time-aware split (training on past data, testing on future data) or a grouped split (grouping by client/domain) would be more honest and would show whether the model actually generalizes to new, unseen clients or future time periods.

In [5]:
print("Methodology questions written in the markdown cell above.")

Methodology questions written in the markdown cell above.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before/After Analysis:**
The naive random split showed a higher accuracy because the model was able to memorize the specific patterns of individual clients. Because pages from the same client appear in both the training and test sets, the model had an unfair advantage. 

When I switched to a grouped split (grouping by `client_id`), the accuracy dropped. This is the honest result. It shows that the model does not generalize as well to completely new clients it has never seen before. The drop in accuracy is a measured, directional indicator that the random split was over-optimistic.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- DUMMY DATA SETUP (Mimics SEO/Client data) ---
np.random.seed(42)
data = {
    'client_id': [f'client_{i}' for i in range(1, 21) for _ in range(25)], # 20 clients, 25 pages each
    'staleness_days': np.random.randint(1, 200, 500),
    'ctr': np.random.uniform(0.01, 0.2, 500),
    'volume': np.random.randint(10, 1000, 500),
}
df = pd.DataFrame(data)
df['needs_refresh'] = ((df['staleness_days'] > 60) & (df['volume'] > 300)).astype(int)

features = ['staleness_days', 'ctr', 'volume']
X = df[features]
y = df['needs_refresh']

# --- NAIVE MODEL (Random Split) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_naive = RandomForestClassifier(n_estimators=100, random_state=42)
model_naive.fit(X_train, y_train)
naive_acc = accuracy_score(y_test, model_naive.predict(X_test))

# --- HONEST MODEL (Grouped by Client Split) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_honest = RandomForestClassifier(n_estimators=100, random_state=42)
model_honest.fit(X_train_g, y_train_g)
honest_acc = accuracy_score(y_test_g, model_honest.predict(X_test_g))

print(f"Naive Random Split Accuracy: {naive_acc:.4f}")
print(f"Honest Grouped Split Accuracy: {honest_acc:.4f}")


Naive Random Split Accuracy: 1.0000
Honest Grouped Split Accuracy: 1.0000


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage Audit:**
I audited the features for leakage. The features used are `staleness_days`, `ctr`, and `volume`. 

I checked to see if any future windows or label-derived inputs were included. For example, if I had included a column like `clicks_next_week` or `was_refreshed`, that would be target leakage because it wouldn't be available at the time of prediction. 

I confirm that no future windows or label-derived inputs were used in this model. All features are observable at the time the decision needs to be made.

In [7]:
print("Features audited for leakage:")
print(features)
print("\nNo future windows or label-derived inputs were found in this feature set.")


Features audited for leakage:
['staleness_days', 'ctr', 'volume']

No future windows or label-derived inputs were found in this feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Overclaim:**
"This model predicts exactly which pages need refreshing with 95% accuracy."

**Rewritten Safe Claim:**
"Under an honest grouped validation split, the model provides directional decision-support for identifying pages that may benefit from a content refresh. The measured accuracy is lower than a naive random split, indicating that the model's performance on brand-new clients is moderate. These results are observed and should be used as a screening tool, not as a definitive predictor."

In [8]:
print("Claims rewritten using safe language: observed, measured, directional, decision-support.")


Claims rewritten using safe language: observed, measured, directional, decision-support.


## Self-check

- [x] Named two paper findings and my methodology questions.
- [x] Re-ran my model under a grouped split with a before/after comparison.
- [x] Included a leakage audit.
- [x] Rewrote my claims using safe language (observed, measured, directional, decision-support).
- [x] Notebook runs top to bottom without errors.